# Cierre del curso: un texto, tres herramientas

---

A lo largo del curso se han introducido tres familias de herramientas. Cada una opera en un nivel de abstracción distinto sobre el mismo objeto: el texto.

| Herramienta | Nivel de análisis | Pregunta que responde |
|---|---|---|
| Regex | Caracteres y patrones formales | ¿Esta cadena tiene esta forma? |
| PLN (spaCy) | Unidades lingüísticas con función | ¿Qué es este token? ¿A qué se refiere? |
| IA generativa | Discurso y contexto | ¿Qué significa esto en este contexto? |

Esta sesión toma un único texto y lo analiza con las tres. El objetivo no es aprender nada nuevo: es ver qué detecta cada herramienta, qué le escapa y por qué.

---

## El texto

El fragmento siguiente procede del comunicado oficial de la Comisión Europea tras la aprobación del Acta de Inteligencia Artificial (2024). Es un texto real, de registro institucional, con terminología técnica y jurídica.

In [ ]:
texto = """
En febrero de 2024, la Comisión Europea aprobó el Acta de Inteligencia Artificial,
el primer marco regulatorio mundial para sistemas de IA de alto riesgo.
El texto fue redactado originalmente en inglés y traducido a las 24 lenguas oficiales
de la Unión por la Dirección General de Traducción.
Los sistemas de traducción automática clasificados como de alto riesgo quedan sujetos
a auditorías periódicas y a la supervisión humana obligatoria.
Para el profesional de la lengua, este contexto implica una transformación del perfil
de competencias requerido: ya no basta con traducir, sino con evaluar, revisar
y validar el output de sistemas automáticos.
"""

print(texto)

---

## 1. Regex

Regex ve el texto como una secuencia de caracteres. Puede extraer todo lo que tenga una forma predecible: fechas, números, siglas, patrones de escritura.

In [ ]:
import re

meses = r'\b(?:enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)\b'
patron_fecha = meses + r' de \d{4}'
patron_numero = r'\b\d+\b'
patron_sigla = r'\b[A-Z]{2,}\b'

print("Fechas:")
for m in re.finditer(patron_fecha, texto, re.IGNORECASE):
    print(f"  '{m.group()}'")

print("\nNúmeros:")
for m in re.finditer(patron_numero, texto):
    print(f"  '{m.group()}'")

print("\nSiglas:")
for m in re.finditer(patron_sigla, texto):
    print(f"  '{m.group()}'")

**Observación:** ¿Qué ha encontrado regex? ¿Qué no ha encontrado?

En particular: ¿puede regex saber que *Comisión Europea* es una institución? ¿Que *traducción automática* es un término técnico? ¿Que *output* es un anglicismo?

---

## 2. PLN (spaCy)

spaCy analiza el texto en términos lingüísticos: categoría gramatical de cada token, tipo de entidad, estructura de la frase. Puede responder preguntas que regex no puede formular.

In [ ]:
!python -m spacy download es_core_news_sm --quiet
import spacy
import pandas as pd

procesador = spacy.load("es_core_news_sm")
documento = procesador(texto)
print("Modelo cargado.")

In [ ]:
print(f"{'Entidad':<40} {'Tipo':<12} {'Descripción'}")
print("-" * 72)
for entidad in documento.ents:
    print(f"{entidad.text:<40} {entidad.label_:<12} {spacy.explain(entidad.label_)}")

In [ ]:
candidatos = []

for i in range(len(documento)):
    token = documento[i]

    if i + 1 < len(documento):
        siguiente = documento[i + 1]
        if token.pos_ == "NOUN" and siguiente.pos_ == "ADJ":
            candidatos.append(f"{token.text} {siguiente.text}")

    if i + 2 < len(documento):
        prep = documento[i + 1]
        segundo = documento[i + 2]
        if token.pos_ == "NOUN" and prep.pos_ == "ADP" and segundo.pos_ == "NOUN":
            candidatos.append(f"{token.text} {prep.text} {segundo.text}")

print("Candidatos a términos complejos:")
for c in sorted(set(candidatos)):
    print(f"  - {c}")

**Observación:** ¿Qué ha encontrado spaCy que regex no podía encontrar?

¿Hay entidades mal clasificadas? ¿Hay términos técnicos relevantes que no han aparecido como candidatos? ¿Por qué podría ser?

---

## 3. IA generativa

Se envía el texto a un modelo de lenguaje con la siguiente instrucción. Se usa cualquiera de las interfaces habituales: [Gemini](https://gemini.google.com), [ChatGPT](https://chatgpt.com), [Claude](https://claude.ai).

---

**Prompt:**

> *El texto siguiente es un fragmento de un comunicado institucional de la Comisión Europea. Identifica los tres términos o expresiones que presentan mayor dificultad para su traducción al inglés. Para cada uno, explica el problema terminológico concreto y propón una traducción justificada.*
>
> *[pegar el texto]*

**Modelo utilizado:**

> *[Gemini / ChatGPT / Claude / otro]*

**Conversación:**

> *[Enlace]*

**Observación:**

> *[¿Qué términos identificó el modelo? ¿Coinciden con los que spaCy detectó como candidatos? ¿La justificación es sólida o genérica?]*

---

## Cierre

Las tres herramientas han analizado el mismo texto. El resultado es diferente en cada caso porque la pregunta era diferente en cada caso.

| Herramienta | Qué encontró | Qué no encontró |
|---|---|---|
| Regex | | |
| spaCy | | |
| IA generativa | | |

*Completar la tabla a partir de lo observado en las secciones anteriores.*

---

### Reflexión final

Estas herramientas automatizan partes de lo que hace un profesional de la lengua. La pregunta que queda abierta es cuál es la parte que no pueden automatizar, y si esa parte es suficiente para sostener una carrera profesional.

El dominio técnico de estas herramientas no es lo que diferencia a un profesional en el mercado. Lo que diferencia es saber qué pregunta se le plantea a cada una, y cuándo la respuesta automática no es suficiente.

Antes del curso, el texto era opaco: se veía lo que decía. Ahora hay tres niveles de lectura más. La pregunta es qué implica eso para la práctica profesional.